# AlphaEarth-System — Linhe PEFT Training on Colab

Trains Prithvi + PEFT adapters on Linhe RGB patches for two tasks:
- **1.a** Building extraction (2-class, synth label)
- **1.b** Land-use classification (6-class, Esri LULC label)

**Before running:**
1. Upload to Drive root: `linhe_2025_h1.tar.gz`, `linhe_2025_h2.tar.gz` (+ `.sha256`), `linhe_lulc_masks.tar.gz`
2. Runtime -> Change runtime type -> L4 GPU
3. Run cells top-to-bottom (skip cell 8 if 1.a already done)

In [ ]:
# 1. Mount Drive, pick archives
from google.colab import drive
drive.mount('/content/drive')

# Two-half packaging keeps each tar.gz under 500 MB so Drive 1 GB free tier
# can hold both simultaneously. Order does not matter — the second tar's
# _index.parquet replaces the first; we merge below in cell 5.
ARCHIVES = [
    'linhe_2025_h1.tar.gz',   # Q1 + Q2  (~420 MB)
    'linhe_2025_h2.tar.gz',   # Q3 + Q4  (~408 MB)
]
CHECKPOINT_DIR = '/content/drive/MyDrive/linhe_ckpt'
RESULTS_DIR = '/content/drive/MyDrive/linhe_results'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted; checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
# 2. GPU check
!nvidia-smi | head -20

In [ ]:
# 3. Clone repo (or pull if already cloned)
%cd /content
REPO_URL = 'https://github.com/zhouning/alphaearth-training-system.git'
REPO_DIR = '/content/alphaearth-training-system'
import os, subprocess
if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!git log --oneline -5

In [ ]:
# 4. Install deps
!pip install -q -e .
!pip install -q pyarrow scikit-learn matplotlib

In [ ]:
# 5. Extract RGB patches + LULC masks, merge indexes
import shutil, subprocess, os
import pandas as pd

# --- RGB patches (h1 + h2) ---
merged_idx_rows, merged_osm_rows = [], []
for arc in ARCHIVES:
    src = f'/content/drive/MyDrive/{arc}'
    sha_src = src + '.sha256'
    dst = f'/content/{arc}'
    assert os.path.exists(src), f'Upload {arc} to Drive first'
    shutil.copy(src, dst)
    shutil.copy(sha_src, '/content/')
    subprocess.run(['sed', '-i', 's/\r$//', f'/content/{os.path.basename(sha_src)}'], check=True)
    subprocess.run(['sha256sum', '-c', f'/content/{os.path.basename(sha_src)}'],
                   cwd='/content', check=True)
    subprocess.run(['tar', 'xzf', dst, '-C', REPO_DIR + '/'], check=True)
    merged_idx_rows.append(pd.read_parquet('data/linhe_patches/_index.parquet'))
    merged_osm_rows.append(pd.read_parquet('data/linhe_patches/_osm_index.parquet'))
    print(f'  extracted {arc}')

merged_idx = pd.concat(merged_idx_rows, ignore_index=True).drop_duplicates('patch_path')
merged_osm = pd.concat(merged_osm_rows, ignore_index=True).drop_duplicates('patch_path')
# Normalize RGB index paths too
for col in ['patch_path']:
    if col in merged_idx.columns:
        merged_idx[col] = merged_idx[col].str.replace('\\', '/', regex=False)
for col in ['patch_path', 'label_path']:
    if col in merged_osm.columns:
        merged_osm[col] = merged_osm[col].str.replace('\\', '/', regex=False)
merged_idx.to_parquet('data/linhe_patches/_index.parquet')
merged_osm.to_parquet('data/linhe_patches/_osm_index.parquet')
rgb_paths = set(merged_idx['patch_path'])
print(f'RGB patches: {len(merged_idx)} patches, {merged_idx["scene_id"].nunique()} scenes')

# --- LULC masks (1.b) ---
LULC_ARC = 'linhe_lulc_masks.tar.gz'
lulc_src = f'/content/drive/MyDrive/{LULC_ARC}'
if os.path.exists(lulc_src):
    shutil.copy(lulc_src, f'/content/{LULC_ARC}')
    subprocess.run(['tar', 'xzf', f'/content/{LULC_ARC}', '-C', 'data/linhe_patches/'], check=True)
    lulc_idx = pd.read_parquet('data/linhe_patches/_lulc_index.parquet')
    # Normalize paths
    for col in ['patch_path', 'lulc_path']:
        if col in lulc_idx.columns:
            lulc_idx[col] = lulc_idx[col].str.replace('\\', '/', regex=False)
    # Filter to only LULC rows whose RGB patch is in the Colab subset
    n_before = len(lulc_idx)
    lulc_idx = lulc_idx[lulc_idx['patch_path'].isin(rgb_paths)].reset_index(drop=True)
    n_after = len(lulc_idx)
    lulc_idx.to_parquet('data/linhe_patches/_lulc_index.parquet')
    print(f'LULC masks: filtered {n_before} -> {n_after} rows (matching RGB subset)')
    print(f'  years: {sorted(lulc_idx["year"].unique())}')
    # Sanity check: pick a sample that should exist
    sample = lulc_idx.iloc[0]
    assert os.path.exists(sample['patch_path']), f'MISSING RGB: {sample["patch_path"]}'
    assert os.path.exists(sample['lulc_path']), f'MISSING LULC: {sample["lulc_path"]}'
    print(f'  sanity check passed (sample: {sample["scene_id"]} year={sample["year"]})')
else:
    print(f'WARNING: {LULC_ARC} not on Drive, 1.b will not work')

In [ ]:
# 6. Download Prithvi weights from HuggingFace (350 MB)
import os
os.makedirs('data/weights/prithvi', exist_ok=True)
if not os.path.exists('data/weights/prithvi/Prithvi_100M.pt'):
    !wget -q https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M/resolve/main/Prithvi_100M.pt -O data/weights/prithvi/Prithvi_100M.pt
!ls -la data/weights/prithvi/

In [ ]:
# 7. GPU throughput sanity check (~30 s)
!python scripts/bench_prithvi_throughput.py --steps 20 --batch-size 16

In [ ]:
# 8. Run 1.a buildings benchmark (synth baseline)
CONFIG_1A = 'geoadapter/bench/configs/linhe_buildings.yaml'
OUTPUT_1A = f'{RESULTS_DIR}/linhe_buildings.json'
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1A} \
    --output {OUTPUT_1A} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --checkpoint-every 2

In [ ]:
# 9. Run 1.b LULC 6-class segmentation benchmark
CONFIG_1B = 'geoadapter/bench/configs/linhe_lulc.yaml'
OUTPUT_1B = f'{RESULTS_DIR}/linhe_lulc_seg.json'
CHECKPOINT_DIR_1B = f'{CHECKPOINT_DIR}_lulc'
os.makedirs(CHECKPOINT_DIR_1B, exist_ok=True)
!python -m geoadapter.bench.run_benchmark \
    --config {CONFIG_1B} \
    --output {OUTPUT_1B} \
    --checkpoint-dir {CHECKPOINT_DIR_1B} \
    --checkpoint-every 2

In [ ]:
# 10. Peek results
import json, os
import pandas as pd

all_results = []
for label, path in [('1.a buildings', OUTPUT_1A), ('1.b LULC', OUTPUT_1B)]:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        for r in data:
            r['task'] = label
        all_results.extend(data)

if all_results:
    df = pd.DataFrame(all_results)
    metric_col = 'mIoU' if 'mIoU' in df.columns else None
    show_cols = ['task', 'method', 'seed', 'trainable_params']
    if metric_col:
        show_cols.append(metric_col)
    show_cols = [c for c in show_cols if c in df.columns]
    print(df[show_cols].sort_values(show_cols[:2], ascending=[True, False]).to_string())
else:
    print('No results yet')

## Notes

- **Resume**: Re-running cell 8 or 9 auto-resumes from last checkpoint and skips completed experiments.
- **1.b only**: Skip cell 8 if 1.a is already done (results on Drive).
- **Clean slate**: Delete `linhe_ckpt/`, `linhe_ckpt_lulc/`, and result JSONs on Drive to force full re-run.
- **Budget**: 1.a + 1.b ~ 250 compute units total.